# Baseline Evaluation of Llama 3.2 1B for Named Entity Recognition (NER)

### Internship Task 1

This notebook evaluates the performance of a pretrained Llama 3.2 1B model on a token classification (NER) task before any fine-tuning.

### Objectives
- Load and inspect the dataset
- Convert the dataset into Hugging Face format
- Perform tokenization and label alignment
- Evaluate the pretrained Llama model
- Report Precision, Recall, F1 Score and Accuracy
- Analyze baseline performance

# 1. Environment Setup

This section installs the required libraries, sets the random seed, loads the tokenizer, and verifies GPU availability.

In [1]:
!pip install -q transformers datasets evaluate seqeval accelerate sentencepiece bitsandbytes peft

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.1 MB/s eta 0:00:00


In [2]:
from huggingface_hub import login

login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [3]:
from transformers import set_seed

set_seed(42)

print("Random seed set to 42")

Random seed set to 42


In [5]:
from transformers import AutoTokenizer

# Llama model used for baseline evaluation
model_name = "meta-llama/Llama-3.2-1B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer loaded successfully!")

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Tokenizer loaded successfully!


In [6]:
import torch

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

GPU: Tesla T4


# 2. Dataset Loading and Inspection

The dataset is stored in CoNLL format where each token is paired with its corresponding NER label.

This section loads the train, validation, and test splits and performs basic dataset inspection.

In [7]:
def read_conll_file(filepath):

    sentences = []
    tags = []

    current_sentence = []
    current_tags = []

    with open(filepath, "r", encoding="utf-8") as f:

        for line in f:

            line = line.strip()

            if not line:

                if current_sentence:
                    sentences.append(current_sentence)
                    tags.append(current_tags)

                    current_sentence = []
                    current_tags = []

            else:

                splits = line.split()

                current_sentence.append(splits[0])
                current_tags.append(splits[-1])

    if current_sentence:
        sentences.append(current_sentence)
        tags.append(current_tags)

    return sentences, tags

In [8]:
train_sentences, train_tags = read_conll_file("train.txt")
valid_sentences, valid_tags = read_conll_file("valid.txt")
test_sentences, test_tags = read_conll_file("test.txt")

print("Train:", len(train_sentences))
print("Validation:", len(valid_sentences))
print("Test:", len(test_sentences))

Train: 14006
Validation: 1717
Test: 1749


# 3. Label Processing

The NER labels are converted into numerical IDs so that they can be used by the neural network model.

In [9]:
all_labels = sorted(
    set(
        label
        for tags in train_tags + valid_tags + test_tags
        for label in tags
    )
)

label2id = {label: idx for idx, label in enumerate(all_labels)}
id2label = {idx: label for label, idx in label2id.items()}

print("Labels:")
print(all_labels)

Labels:
['B-LONG', 'B-SHORT', 'B-long', 'B-short', 'I-LONG', 'I-SHORT', 'I-long', 'I-short', 'O']


# 4. Hugging Face Dataset Creation

The raw Python lists are converted into Hugging Face Dataset objects for efficient processing and evaluation.

In [10]:
from datasets import Dataset, DatasetDict

dataset = DatasetDict({
    "train": Dataset.from_dict({
        "tokens": train_sentences,
        "ner_tags": train_tags
    }),

    "validation": Dataset.from_dict({
        "tokens": valid_sentences,
        "ner_tags": valid_tags
    }),

    "test": Dataset.from_dict({
        "tokens": test_sentences,
        "ner_tags": test_tags
    })
})

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 14006
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 1717
    })
    test: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 1749
    })
})


In [11]:
print(dataset["train"][0])

{'tokens': ['What', 'is', 'here', 'called', 'controlled', 'natural', 'language', '(', 'CNL', ')', 'has', 'traditionally', 'been', 'given', 'many', 'different', 'names', '.'], 'ner_tags': ['O', 'O', 'O', 'O', 'B-long', 'I-long', 'I-long', 'O', 'B-short', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']}


# 5. Tokenization and Label Alignment

Llama tokenizes words into subword units.

To preserve correct NER annotations:
- Special tokens are ignored using -100
- Only the first subword receives the entity label
- Remaining subwords are masked with -100

In [12]:
def tokenize_and_align_labels(examples):

    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []

    for i, label in enumerate(examples["ner_tags"]):

        word_ids = tokenized_inputs.word_ids(batch_index=i)

        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:

            if word_idx is None:
                label_ids.append(-100)

            elif word_idx != previous_word_idx:
                label_ids.append(label2id[label[word_idx]])

            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels

    return tokenized_inputs

In [13]:
tokenized_dataset = dataset.map(
    tokenize_and_align_labels,
    batched=True
)

print(tokenized_dataset)

Map:   0%|          | 0/14006 [00:00<?, ? examples/s]

Map:   0%|          | 0/1717 [00:00<?, ? examples/s]

Map:   0%|          | 0/1749 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 14006
    })
    validation: Dataset({
        features: ['tokens', 'ner_tags', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 1717
    })
    test: Dataset({
        features: ['tokens', 'ner_tags', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 1749
    })
})


# 6. Model Loading

A pretrained Llama 3.2 1B model is loaded for token classification.

No fine-tuning is performed in this notebook.
The goal is to establish baseline performance.

In [14]:
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(all_labels),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

model.config.pad_token_id = tokenizer.pad_token_id

print("Model loaded successfully!")

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

[transformers] LlamaForTokenClassification LOAD REPORT from: meta-llama/Llama-3.2-1B-Instruct
Key          | Status  | 
-------------+---------+-
score.bias   | MISSING | 
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded successfully!


In [15]:
total_params = sum(p.numel() for p in model.parameters())

print("Model Name:", model_name)
print(f"Total Parameters: {total_params:,}")

Model Name: meta-llama/Llama-3.2-1B-Instruct
Total Parameters: 1,235,832,841


# 7. Evaluation Metrics

Performance is evaluated using:

- Precision
- Recall
- F1 Score
- Accuracy

The SeqEval library is used because it is designed specifically for sequence labeling tasks such as NER.

In [16]:
import numpy as np
import evaluate

seqeval = evaluate.load("seqeval")

def compute_metrics(p):

    predictions, labels = p

    predictions = np.argmax(predictions, axis=2)

    true_predictions = []
    true_labels = []

    for prediction, label in zip(predictions, labels):

        pred_labels = []
        true_lbls = []

        for p, l in zip(prediction, label):

            if l != -100:
                pred_labels.append(id2label[p])
                true_lbls.append(id2label[l])

        true_predictions.append(pred_labels)
        true_labels.append(true_lbls)

    results = seqeval.compute(
        predictions=true_predictions,
        references=true_labels
    )

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"]
    }

In [17]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer
)

# 8. Baseline Evaluation

The pretrained Llama model is evaluated on the test dataset without any fine-tuning.

These results serve as the baseline for comparison with future fine-tuned models.

In [18]:
from transformers import TrainingArguments, Trainer

baseline_args = TrainingArguments(
    output_dir="./baseline_eval",
    per_device_eval_batch_size=8,
    report_to="none"
)

baseline_trainer = Trainer(
    model=model,
    args=baseline_args,
    eval_dataset=tokenized_dataset["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

baseline_results = baseline_trainer.evaluate()

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Training Loss,Validation Loss,Step,Precision,Recall,F1,Accuracy
No log,2.125075,0,0.005391,0.095727,0.010207,0.281855


# 9. Results Analysis

This section summarizes the baseline performance metrics.

In [19]:
import pandas as pd

results_df = pd.DataFrame({
    "Metric": ["Precision", "Recall", "F1", "Accuracy"],
    "Score": [
        baseline_results["eval_precision"],
        baseline_results["eval_recall"],
        baseline_results["eval_f1"],
        baseline_results["eval_accuracy"]
    ]
})

results_df

,Metric,Score
0,Precision,0.005391
1,Recall,0.095727
2,F1,0.010207
3,Accuracy,0.281855


In [20]:
import pandas as pd

results_df = pd.DataFrame({
    "Metric": ["Precision", "Recall", "F1", "Accuracy"],
    "Score": [
        baseline_results["eval_precision"],
        baseline_results["eval_recall"],
        baseline_results["eval_f1"],
        baseline_results["eval_accuracy"]
    ]
})

results_df

,Metric,Score
0,Precision,0.005391
1,Recall,0.095727
2,F1,0.010207
3,Accuracy,0.281855


# 10. Per-Entity Performance Analysis

A detailed classification report is generated to analyze model performance across different entity categories.

In [21]:
from seqeval.metrics import classification_report

predictions, labels, _ = baseline_trainer.predict(
    tokenized_dataset["test"]
)

predictions = np.argmax(predictions, axis=2)

true_predictions = [
    [id2label[p] for p, l in zip(prediction, label) if l != -100]
    for prediction, label in zip(predictions, labels)
]

true_labels = [
    [id2label[l] for p, l in zip(prediction, label) if l != -100]
    for prediction, label in zip(predictions, labels)
]

print(classification_report(true_labels, true_predictions))

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


              precision    recall  f1-score   support

        LONG       0.00      0.00      0.00        25
       SHORT       0.02      0.10      0.03      1824
        long       0.00      0.00      0.00         0
       short       0.00      0.00      0.00         0

   micro avg       0.01      0.10      0.01      1849
   macro avg       0.01      0.02      0.01      1849
weighted avg       0.02      0.10      0.03      1849



# Conclusion

The pretrained Llama 3.2 1B model was evaluated on the NER dataset without any task-specific fine-tuning.

The resulting Precision, Recall, F1 Score, and Accuracy establish a baseline benchmark that can be used for comparison with future fine-tuning experiments.

Future work:
- LoRA Fine-Tuning
